**Prajakta Ghorpade SCFP125004**
Lab3: Prompt_Engineering_Exercises_1_to_6

In [3]:

!pip install -q langchain-groq langchain-core python-dotenv

from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY
)

parser = StrOutputParser()
json_parser = JsonOutputParser()

print("Setup completed successfully!")



Setup completed successfully!


In [4]:

response = llm.invoke("Say hello in one sentence.")
print(response.content)


Hello!


Exercise 1: Doctor Summary Generator

In [5]:

patient_name = "Prajakta Ghorpade"

patient_notes = """
Patient has cough and mild fever for 3 days.
Doctor advised rest and fluids.
"""

doctor_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a medical summary assistant.
Use only the given patient notes.
Do not invent medical information.
If diagnosis is missing, write Not provided.
Return only these sections:
Symptoms, Diagnosis, Recommendation."""),
    ("user", """Patient Name: {patient_name}
Patient Notes: {patient_notes}""")
])

doctor_chain = doctor_prompt | llm | parser

response = doctor_chain.invoke({
    "patient_name": patient_name,
    "patient_notes": patient_notes
})

print(response)


**Symptoms**  
- Cough  
- Mild fever for 3 days  

**Diagnosis**  
Not provided  

**Recommendation**  
- Rest  
- Increase fluid intake  


Exercise 2: Review Classifier

In [6]:

review_text = "The product is good and works perfectly."

review_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the review into one label. Positive means satisfied, Negative means dissatisfied, and Neutral means no clear opinion. Return only Positive, Negative, or Neutral."),
    ("user", "Customer review: {review_text}")
])

review_chain = review_prompt | llm | parser

response = review_chain.invoke({
    "review_text": review_text
})

print(response)


Positive


Exercise 3: Resume Field Extractor

In [7]:

resume_text = """
Name: Ghorpade Prajakta
Skills: Python, Java, SQL
Experience: 2 years
Email: praj1909@gmail.com
"""

fields_to_extract = ["name", "skills", "email"]

resume_prompt = ChatPromptTemplate.from_messages([
    ("system", """Extract only the requested fields from the resume.
Return valid JSON only, without Markdown or extra text.
Use null for missing information. Do not guess.
{format_instructions}"""),
    ("user", "Resume: {resume_text}\nFields: {fields_to_extract}")
]).partial(
    format_instructions=json_parser.get_format_instructions()
)

resume_chain = resume_prompt | llm | json_parser

response = resume_chain.invoke({
    "resume_text": resume_text,
    "fields_to_extract": fields_to_extract
})

print(response)


{'name': 'Ghorpade Prajakta', 'skills': ['Python', 'Java', 'SQL'], 'email': 'praj1909@gmail.com'}


Exercise 4: Customer Support Reply Generator

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [9]:



sentence = "Could you please help me with this problem?"
target_language = "Marathi"
formality = "formal"

translation_prompt = ChatPromptTemplate.from_messages([
    ("system", """Translate the sentence into {target_language}.
Use a {formality} formality level.
Keep the original meaning.
Return only the translated sentence."""),
    ("user", "Sentence: {sentence}")
])

translation_chain = translation_prompt | llm | parser

response = translation_chain.invoke({
    "sentence": sentence,
    "target_language": target_language,
    "formality": formality
})

print("Original Sentence:", sentence)
print("Translated Sentence:", response)



Original Sentence: Could you please help me with this problem?
Translated Sentence: कृपया आपण मला या समस्येचे निराकरण करण्यात मदत करू शकता का?


Exercise 5: Multi-language Translator

In [10]:

sentence = "Could you please help me with this problem?"
target_language = "Marathi"
formality = "formal"

translation_prompt = ChatPromptTemplate.from_messages([
    ("system", """Translate the sentence into {target_language}.
Use a {formality} formality level.
Keep the original meaning.
Return only the translated sentence."""),
    ("user", "Sentence: {sentence}")
])

translation_chain = translation_prompt | llm | parser

response = translation_chain.invoke({
    "sentence": sentence,
    "target_language": target_language,
    "formality": formality
})

print(response)


कृपया आपण मला या समस्येत मदत करू शकाल का?


Exercise 6: Iterative Content Refiner

In [11]:

draft_text = "our app is good it helps students manage work"
issue_to_fix = "grammar and punctuation"

refine_prompt = ChatPromptTemplate.from_messages([
    ("system", """Revise the given text to fix only this issue:
{issue_to_fix}
Keep the original meaning.
Do not add new ideas.
Return only the revised text."""),
    ("user", "Draft: {draft_text}")
])

refine_chain = refine_prompt | llm | parser

round_1 = refine_chain.invoke({
    "draft_text": draft_text,
    "issue_to_fix": issue_to_fix
})
print("Round 1:", round_1)

round_2 = refine_chain.invoke({
    "draft_text": round_1,
    "issue_to_fix": issue_to_fix
})
print("Round 2:", round_2)

round_3 = refine_chain.invoke({
    "draft_text": round_2,
    "issue_to_fix": issue_to_fix
})
print("Round 3:", round_3)


Round 1: Draft: Our app is good. It helps students manage work.
Round 2: Draft: Our app is good. It helps students manage work.
Round 3: Draft: Our app is good. It helps students manage work.
